In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [2]:
DATA_PATH = Path("data/processed/telco_features.csv")
MODEL_PATH = Path("models/best_model.joblib")
REPORT_DIR = Path("reports/xai")

REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

In [3]:
X = df.drop(columns=["Churn"], errors="ignore")
y = df["Churn"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = joblib.load(MODEL_PATH)

In [5]:
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["model"]


In [6]:
X_background = X_train.sample(
    n=min(100, len(X_train)),
    random_state=42
)

X_explain = X_test.sample(
    n=min(200, len(X_test)),
    random_state=42
)

In [7]:
background_transformed = preprocessor.transform(X_background)
explain_transformed = preprocessor.transform(X_explain)

# Noms des features après preprocessing
feature_names = preprocessor.get_feature_names_out()

background_df = pd.DataFrame(
    background_transformed,
    columns=feature_names
)

explain_df = pd.DataFrame(
    explain_transformed,
    columns=feature_names
)

In [8]:
explainer = shap.Explainer(
    classifier.predict_proba,
    background_df
)

shap_values = explainer(explain_df)



PermutationExplainer explainer: 201it [00:19,  3.55it/s]                         


In [9]:
values = shap_values.values

if values.ndim == 3:
    churn_shap = values[:, :, 1]
    base_values = shap_values.base_values

    if base_values.ndim == 2:
        churn_base = base_values[:, 1]
    else:
        churn_base = base_values
else:
    churn_shap = values
    churn_base = shap_values.base_values

global_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": np.abs(churn_shap).mean(axis=0)
}).sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

print("\n===== IMPORTANCE GLOBALE SHAP =====")
print(global_importance.head(15).to_string(index=False))

global_importance.to_csv(
    REPORT_DIR / "global_feature_importance.csv",
    index=False
)



===== IMPORTANCE GLOBALE SHAP =====
                            Feature  Mean_Absolute_SHAP
       num__Contract_Month-to-month            0.080029
                        num__tenure            0.062752
                num__MonthlyCharges            0.042193
   num__InternetService_Fiber optic            0.041934
                  num__TotalCharges            0.039453
                num__TechSupport_No            0.027760
num__PaymentMethod_Electronic check            0.023089
                num__OnlineSecurity            0.020975
                 num__MultipleLines            0.018859
              num__PaperlessBilling            0.017977
             num__Contract_Two year            0.015826
                  num__PhoneService            0.012280
                 num__SeniorCitizen            0.008830
                    num__Dependents            0.008435
           num__InternetService_DSL            0.008234


In [10]:
plt.figure()
shap.summary_plot(
    churn_shap,
    explain_df,
    show=False,
    max_display=15
)
plt.tight_layout()
plt.savefig(
    REPORT_DIR / "shap_summary.png",
    bbox_inches="tight"
)
plt.close()


C:\Users\eyadh\AppData\Local\Temp\ipykernel_3700\804174968.py:2: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


In [11]:
idx = 0

local_explanation = pd.DataFrame({
    "Feature": feature_names,
    "Value": explain_df.iloc[idx].values,
    "SHAP_Value": churn_shap[idx]
})

local_explanation["Abs_SHAP"] = (
    local_explanation["SHAP_Value"].abs()
)

local_explanation = local_explanation.sort_values(
    "Abs_SHAP",
    ascending=False
)

print("\n===== EXPLICATION LOCALE =====")
print(local_explanation.head(10).to_string(index=False))

local_explanation.to_csv(
    REPORT_DIR / "local_explanation.csv",
    index=False
)



===== EXPLICATION LOCALE =====
                            Feature     Value  SHAP_Value  Abs_SHAP
   num__InternetService_Fiber optic -0.887696   -0.060197  0.060197
       num__Contract_Month-to-month  0.903464    0.042745  0.042745
              num__PaperlessBilling -1.202653   -0.038801  0.038801
                        num__tenure -0.955978    0.038279  0.038279
           num__InternetService_DSL  1.381529   -0.025282  0.025282
                num__MonthlyCharges -0.467223   -0.024682  0.024682
                  num__OnlineBackup  1.359534   -0.020835  0.020835
num__PaymentMethod_Electronic check -0.710781   -0.020166  0.020166
                num__OnlineSecurity -0.636111    0.017389  0.017389
                 num__SeniorCitizen -0.441773   -0.011757  0.011757


In [12]:
explanation = shap.Explanation(
    values=churn_shap[idx],
    base_values=(
        churn_base[idx]
        if np.ndim(churn_base) > 0
        else churn_base
    ),
    data=explain_df.iloc[idx].values,
    feature_names=feature_names
)

shap.plots.waterfall(explanation, show=False)
plt.tight_layout()
plt.savefig(
    REPORT_DIR / "shap_waterfall.png",
    bbox_inches="tight"
)
plt.close()

In [13]:
client_probability = model.predict_proba(
    X_explain.iloc[[idx]]
)[0, 1]

print(f"\nProbabilité prédite de churn : {client_probability:.2%}")
print("Classe prédite :", "Churn" if client_probability >= 0.5 else "No churn")

print(f"\nRésultats sauvegardés dans : {REPORT_DIR}")



Probabilité prédite de churn : 15.88%
Classe prédite : No churn

Résultats sauvegardés dans : reports\xai


In [15]:
from lime.lime_tabular import LimeTabularExplainer

# Convertir les données transformées en tableaux denses
# si le préprocesseur renvoie une matrice sparse
if hasattr(background_transformed, "toarray"):
    background_array = background_transformed.toarray()
else:
    background_array = np.asarray(background_transformed)

if hasattr(explain_transformed, "toarray"):
    explain_array = explain_transformed.toarray()
else:
    explain_array = np.asarray(explain_transformed)

feature_names_list = list(feature_names)

# Créer l'explicateur LIME
lime_explainer = LimeTabularExplainer(
    training_data=background_array,
    feature_names=feature_names_list,
    class_names=["No churn", "Churn"],
    mode="classification",
    discretize_continuous=True,
    random_state=42
)

# Choisir le même client que pour SHAP
idx = 0
client_data = explain_array[idx]

# LIME reçoit des données déjà transformées.
# Le classificateur utilisé est celui du pipeline.
lime_exp = lime_explainer.explain_instance(
    data_row=client_data,
    predict_fn=classifier.predict_proba,
    labels=(1,),
    num_features=10,
    num_samples=3000
)

# Afficher les contributions locales
print("\n===== EXPLICATION LOCALE LIME =====")

for feature, contribution in lime_exp.as_list(label=1):
    print(f"{feature}: {contribution:.4f}")

# Sauvegarder le rapport HTML
lime_exp.save_to_file(
    str(REPORT_DIR / "lime_explanation.html")
)

# Sauvegarder également le graphique
fig = lime_exp.as_pyplot_figure(label=1)
fig.tight_layout()
fig.savefig(
    REPORT_DIR / "lime_explanation.png",
    bbox_inches="tight"
)
plt.close(fig)

print("Explication LIME sauvegardée.")



===== EXPLICATION LOCALE LIME =====
-1.11 < num__Contract_Month-to-month <= 0.90: 0.2237
num__tenure <= -0.96: 0.1163
-0.98 < num__TechSupport_No <= 1.02: 0.0910
num__InternetService_Fiber optic <= -0.89: -0.0806
num__PaymentMethod_Electronic check <= -0.71: -0.0761
num__Contract_Two year <= -0.56: 0.0737
num__OnlineSecurity <= -0.64: 0.0621
num__Dependents <= -0.65: 0.0579
num__PaperlessBilling <= -1.20: -0.0482
num__MultipleLines <= -0.86: -0.0384
Explication LIME sauvegardée.
